# Data Science Project



Index
1. Imports
2. Load Data
3. EDA (Exploratory Data Analysis)
4. Feature Engineering
5. Baseline Models
6. Models
7. Ensembles
8. Deep Learning
9. Evaluation (Comparison)
10. Conclusions




## 1. Imports
Import necessary libraries and set configurations.


In [ ]:
# ===========================================================
# 1. IMPORTS AND INITIAL CONFIGURATION
# ===========================================================

# --- Data manipulation and analysis ---
import pandas as pd
import numpy as np
import gc                        # Garbage collector: manually free RAM

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Scikit-learn: classical ML ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (classification_report,
                              confusion_matrix,
                              f1_score,
                              accuracy_score)

# --- Computer vision: feature extraction ---
from scipy import ndimage         # Sobel filter (edge detection)
from skimage.feature import hog   # Histogram of Oriented Gradients

# --- TensorFlow / Keras: Deep Learning ---
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout

# --- Generative AI ---
!pip install -q transformers sentence-transformers accelerate
# Uncomment the line above if this is the first time you run the notebook
import torch
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# --- Utilities ---
import warnings
import random

# -----------------------------------------------------------
# GLOBAL RANDOM SEED (guarantees full reproducibility)
# -----------------------------------------------------------
SEED = 42
random.seed(SEED)           # Python's random module
np.random.seed(SEED)        # NumPy: shuffles, random samples, etc.
tf.random.set_seed(SEED)    # TensorFlow/Keras: CNN weight initialization

# General configuration
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style("whitegrid")

# -----------------------------------------------------------
# Fashion-MNIST class dictionary and list
# Defined here so they can be used anywhere in the notebook
# -----------------------------------------------------------
class_names_dict = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot"
}
class_names = list(class_names_dict.values())  # List for axes and confusion matrices

# -----------------------------------------------------------
# Path to the data folder
# -----------------------------------------------------------
# Place fashion-mnist_train.csv, fashion-mnist_test.csv, and
# fashion_mnist_metadata.json inside the local "data/" folder.
# If running on Google Colab instead, mount Drive and point this
# to your dataset folder there, e.g.:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   FOLDER_PATH = "/content/drive/MyDrive/<your-folder>/"
FOLDER_PATH = "data/"


## 2. Load Data
Load the train and test CSV files of the Fashion-MNIST dataset from the local data/ folder.


In [ ]:
# ===========================================================
# 2. LOAD DATA
# ===========================================================

# Load the training CSV (60,000 images + labels)
df_train = pd.read_csv(FOLDER_PATH + "fashion-mnist_train.csv")

# Load the test CSV (10,000 images + labels)
df_test = pd.read_csv(FOLDER_PATH + "fashion-mnist_test.csv")

# Concatenate both datasets into one for the full EDA (70,000 samples)
# axis=0 = stack rows | ignore_index=True = reindex from 0
df = pd.concat([df_train, df_test], axis=0, ignore_index=True)

# Show the first 5 rows: 'label' column + 784 pixel columns (pixel1...pixel784)
df.head()


## 3. Exploratory Data Analysis (EDA)
Analyze the data distribution, correlations, and target balance.


In [ ]:
# Dataset dimensions: (70,000 rows, 785 columns)
# 1 'label' column + 784 pixels (flattened 28x28 image)
df.shape


In [ ]:
# General DataFrame info:
# - Data types (all int64: labels and pixels are integers 0-255)
# - Non-null values per column
df.info()


In [ ]:
# Basic descriptive statistics for all numeric columns:
# count, mean, std, min, 25%, 50%, 75%, max
# Useful for detecting anomalies or values outside the expected range (0-255)
df.describe()


In [ ]:
# Check for null values across the whole dataset
# Fashion-MNIST is clean by construction, but it's good practice to verify it
print(df.isnull().sum().sum(), "total null values.\n")


In [ ]:
# Class distribution: check whether the dataset is balanced
# Fashion-MNIST has exactly 7,000 images per class (10 classes x 7,000 = 70,000)
plt.figure(figsize=(10, 5))
sns.countplot(x='label', data=df, palette='viridis')
plt.title("Class distribution - Full dataset")
plt.xticks(ticks=np.arange(10), labels=class_names, rotation=45)
plt.show()

# Exact count per numeric label
df['label'].value_counts()


In [ ]:
# Visualize 25 random images from the dataset to understand what the garments look like
# np.random already has the seed fixed (SEED=42) -> same 25 images on every run
rng = np.random.RandomState(SEED)
plt.figure(figsize=(10, 10))

for i in range(25):
    idx = rng.randint(0, len(df))
    image = df.iloc[idx, 1:].values.reshape(28, 28)
    label = df.iloc[idx, 0]

    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(image, cmap=plt.cm.binary)
    plt.xlabel(class_names_dict[label], fontsize=8)

plt.tight_layout()
plt.show()


**Visual observation:** The classes with the highest risk of confusion are the torso garments: *Coat*, *Shirt*, *Pullover*, and *T-shirt/top* share similar silhouettes with sleeves and a body shape. In contrast, *Trouser*, *Bag*, *Sandal*, and *Ankle Boot* have very distinctive shapes and should be easier to classify correctly.

In [ ]:
# Analysis of pixel intensity distribution
# Extract only the pixel columns (excluding the 'label' column)
pixel_values = df.iloc[:, 1:]

# Extreme values and global mean to understand the data range
print("Minimum value:", pixel_values.min().min())   # Should be 0 (black)
print("Maximum value:", pixel_values.max().max())   # Should be 255 (white)
print("Global mean:", pixel_values.mean().mean())    # Mean of all pixels across all images


In [ ]:
# Histogram of pixel intensity distribution across the whole dataset
# A very high peak at 0 (black) is expected since the image backgrounds are black
plt.figure(figsize=(8, 5))
sns.histplot(pixel_values.values.flatten(), bins=50)
plt.title("Pixel intensity distribution")
plt.show()


**Observation:** The histogram shows a very pronounced peak at value 0 (black pixel), which is expected since all images have a black background. The garment information is concentrated in the medium and high intensity values.

In [ ]:
# Average image per class: compute the mean of all pixels
# for each category separately and visualize it as an image
# This lets us see which shape/silhouette characterizes each garment
plt.figure(figsize=(12, 8))

for label in range(10):
    # Filter the rows for that class and compute the mean of each pixel
    mean_image = df[df['label'] == label].iloc[:, 1:].mean().values.reshape(28, 28)

    plt.subplot(2, 5, label + 1)
    plt.imshow(mean_image, cmap='gray')
    plt.title(class_names[label])
    plt.axis("off")

plt.tight_layout()
plt.show()


**Observation:** The average images confirm that *Shirt*, *Coat*, and *Pullover* are the most visually similar categories: they all show sleeves and a similarly shaped body. This anticipates that they will be the classes with the highest error rate in the models. *Trouser* and *Bag* stand out for their distinctive shape.

In [ ]:
# Correlation matrix between pixels
# Goal: detect whether there are groups of highly correlated pixels
# (Given the high dimensionality 784x784, the result is hard to interpret visually)
corr = pixel_values.corr()
corr


This doesn't add anything useful in this form.

In [ ]:
# Shows the KDE distribution of intensities per class,
# letting us see which classes have darker/lighter pixels on average.
plt.figure(figsize=(10,6))

for label in range(10):
    class_pixels = df[df['label']==label].iloc[:,1:].values.flatten()
    sns.kdeplot(class_pixels, label=class_names[label], fill=False)

plt.legend()
plt.title("Intensity distribution by class")
plt.show()


In [ ]:
# -----------------------------------------------------------
# Spatial variance map per pixel
# -----------------------------------------------------------
# Compute the variance of each of the 784 pixels across all images
# and display it as a 28x28 image.
# Pixels with high variance (hot zones in the map) are the most informative,
# since they change a lot between images -> they carry more discriminative information.
variance_image = df.iloc[:, 1:].var().values.reshape(28, 28)

plt.imshow(variance_image, cmap='hot')
plt.colorbar()
plt.title("Variance per pixel")
plt.show()


In [ ]:
# -----------------------------------------------------------
# 2D PCA visualization (class separability analysis)
# -----------------------------------------------------------
# PCA (Principal Component Analysis) reduces the dimensionality from 784 to 2 components
# so the data can be visualized on a 2D plane.
# If the classes appear well separated in 2D, they are easier to classify.

X = df.iloc[:, 1:] / 255.0   # Normalized pixels
y = df['label']

# We use only 5,000 random samples to make the visualization faster
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.sample(5000, random_state=42))
y_sample = y.sample(5000, random_state=42)

# 2D scatter plot where each point is an image and the color represents the class
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_sample, cmap='tab10', s=5)
plt.colorbar(scatter)
plt.title("PCA 2D projection")
plt.show()


## 4. Feature Engineering
Prepare data for modeling: splitting, scaling, encoding (not needed here as data is numerical).


In [ ]:
# ===========================================================
# 4. FEATURE ENGINEERING
# ===========================================================

# -----------------------------------------------------------
# 4.1 Pixel Normalization (0-255 scale -> 0-1)
# -----------------------------------------------------------
# Why? Many ML algorithms are sensitive to the scale of the data.
# Dividing by 255 standardizes the range to [0, 1], which:
#   - Speeds up convergence for models like SVM, Neural Networks
#   - Prevents pixels with high values from artificially carrying more "weight"

# Separate the target variable (label) from the features (pixels)
X = df.drop('label', axis=1)   # 784 pixel columns
y = df['label']                # Class labels (0-9)

# Divide by 255 to normalize to the [0, 1] range
X_normalized = X / 255.0

# Check: the maximum should now be 1.0
print("Maximum values before normalization:", X.max().max())
print("Maximum values after normalization:", X_normalized.max().max())


The spatial variance reveals that the central area concentrates the most variability, indicating that the discriminative information is mainly located in the garment outlines.

In [ ]:

# -----------------------------------------------------------
# 4.2 Horizontal and Vertical Symmetry as new features
# -----------------------------------------------------------
# Why? Garments like trousers or bags have high horizontal symmetry
# (they look the same on the left and right side), while a sandal or boot does not.
# This symmetry difference helps the models distinguish between categories.

# 1. Convert the flat rows (784 columns) into 28x28 matrices
#    -1 means "as many images as needed" (keeps all 60,000 or 70,000)
images = X.values.reshape(-1, 28, 28)

# 2. Horizontal asymmetry: difference between the image and its left-right mirror
#    np.flip(axis=2) flips the pixels along the horizontal axis (columns)
flipped_images_h = np.flip(images, axis=2)
# Mean absolute pixel-by-pixel difference between the original and the horizontal mirror
horizontal_asymmetry = np.mean(np.abs(images - flipped_images_h), axis=(1, 2))

# 3. Vertical asymmetry: difference between the image and its top-bottom mirror
#    np.flip(axis=1) flips the pixels along the vertical axis (rows)
flipped_images_v = np.flip(images, axis=1)
vertical_asymmetry = np.mean(np.abs(images - flipped_images_v), axis=(1, 2))

# 4. Add the new columns to the normalized dataset
X_fe = X.copy()
X_fe['horizontal_asymmetry'] = horizontal_asymmetry
X_fe['vertical_asymmetry'] = vertical_asymmetry

print("New dataset dimensions:", X_fe.shape)
print(X_fe[['horizontal_asymmetry', 'vertical_asymmetry']].head())


In [ ]:

# -----------------------------------------------------------
# 4.3 Sobel Edge Filter
# -----------------------------------------------------------
# What is the Sobel filter?
#   It's a gradient operator that detects edges in images.
#   It computes the image derivative in the X direction (vertical edges)
#   and in the Y direction (horizontal edges), then combines them.
# Why use it?
#   The outline of a garment (e.g. the straight edge of trousers vs.
#   the curves of a t-shirt) is very discriminative for ML models.

# 1. Reshape the normalized data into 28x28 images
images = X_normalized.values.reshape(-1, 28, 28)

# Empty array to store the filtered images
sobel_images = np.zeros_like(images)

# 2. Apply the Sobel filter image by image
for i in range(images.shape[0]):
    # Gradient along the horizontal axis (detects horizontal edges)
    sobel_x = ndimage.sobel(images[i], axis=0)
    # Gradient along the vertical axis (detects vertical edges)
    sobel_y = ndimage.sobel(images[i], axis=1)
    # Total gradient magnitude (Pythagoras: sqrt(Gx^2 + Gy^2))
    sobel_images[i] = np.hypot(sobel_x, sobel_y)

# 3. Flatten back to 784 columns (format required by classical ML)
X_sobel_flat = sobel_images.reshape(-1, 784)

# 4. Build a DataFrame with descriptive names to avoid confusion with the original pixels
sobel_columns = [f'sobel_pixel{i+1}' for i in range(784)]
df_sobel = pd.DataFrame(X_sobel_flat, columns=sobel_columns)

print("Sobel dataset dimensions:", df_sobel.shape)


In [ ]:

# Visual comparison: original image vs. image with the Sobel filter applied
# This lets us visually verify that the filter correctly detects the edges

index = 1   # Index of the image to visualize

# Original 28x28 image (already in the 'images' array)
original_image = images[index]
# Same image but with the Sobel filter applied
filtered_image = sobel_images[index]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(original_image, cmap='gray')
axes[0].set_title(f'Original Image (Index {index})')
axes[0].axis('off')

axes[1].imshow(filtered_image, cmap='gray')
axes[1].set_title('Sobel Filter (Edges)')
axes[1].axis('off')

plt.show()


In [ ]:

# -----------------------------------------------------------
# 4.4 HOG: Histogram of Oriented Gradients
# -----------------------------------------------------------
# What is HOG?
#   HOG divides the image into small blocks and, for each block, computes
#   a histogram of the gradient orientations (edge angles).
#   Instead of looking at individual pixels, it captures the local SHAPE of the garment.
# Why use it?
#   It is very robust to small variations in lighting and position.
# Chosen parameters:
#   - orientations=9: 9 angle bins (0, 20, 40, ..., 160 degrees)
#   - pixels_per_cell=(14,14): split the 28x28 image into 4 blocks of 14x14
#   - cells_per_block=(1,1): normalize block by block

hog_features_list = []

# Process each image in the dataset (can take 1-2 minutes with 70,000 images)
for img in images:
    features = hog(img, orientations=9, pixels_per_cell=(14, 14),
                   cells_per_block=(1, 1), visualize=False)
    hog_features_list.append(features)

# Convert the list into a DataFrame with descriptive names
df_hog = pd.DataFrame(hog_features_list)
df_hog.columns = [f'hog_feature_{i+1}' for i in range(df_hog.shape[1])]

print("HOG feature dimensions:", df_hog.shape)
df_hog.head()


In [ ]:

# Visual comparison: original image vs. HOG image
# HOG with pixels_per_cell=(4,4) produces a more detailed visualization than the one used in the model
# (we use 4x4 here only to make the visualization clearer)

index = 1
original_image = images[index]

# visualize=True also returns the HOG image so it can be plotted
features, hog_image = hog(original_image, orientations=9, pixels_per_cell=(4, 4),
                          cells_per_block=(1, 1), visualize=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(original_image, cmap='gray')
axes[0].set_title(f'Original Image (Index {index})')
axes[0].axis('off')

axes[1].imshow(hog_image, cmap='gray')
axes[1].set_title('HOG (Histogram of Oriented Gradients)')
axes[1].axis('off')

plt.show()


In [ ]:
# -----------------------------------------------------------
# 4.5 Intensity Histograms and Global Statistics
# -----------------------------------------------------------
# We create three groups of new manual features:
#
# (A) Intensity histogram (16 bins):
#     Counts how many pixels fall in each brightness range.
#     E.g. a black t-shirt will have many pixels in the low bins.
#
# (B) Global statistics:
#     - Mean: average brightness of the whole image
#     - Standard deviation: brightness variability (a uniform image has low std)
#     - Energy: sum of squares (sensitive to very bright pixels)
#
# (C) Mean per quadrant (14x14 pixels):
#     Split the image into 4 zones and compute the mean brightness of each.
#     Captures basic spatial information: where is most of the garment located?

# (A) Intensity histograms
histograms = []
for img in images:
    # 16 bins between 0 and 1 (pixels are already normalized)
    hist, _ = np.histogram(img, bins=16, range=(0, 1))
    histograms.append(hist)

hist_columns = [f'hist_bin_{i+1}' for i in range(16)]
df_histograms = pd.DataFrame(histograms, columns=hist_columns)

# (B) Global statistics per image
# axis=(1,2) applies the operation to each image individually (not to the whole array)
global_mean = np.mean(images, axis=(1, 2))
global_std  = np.std(images, axis=(1, 2))
energy = np.sum(images**2, axis=(1, 2))

# (C) Mean per quadrant (split the 28x28 image into 4 zones of 14x14)
top_left     = np.mean(images[:, :14, :14], axis=(1, 2))
top_right    = np.mean(images[:, :14, 14:], axis=(1, 2))
bottom_left  = np.mean(images[:, 14:, :14], axis=(1, 2))
bottom_right = np.mean(images[:, 14:, 14:], axis=(1, 2))

# Assemble all the new features into a single DataFrame
df_custom_features = pd.DataFrame({
    'global_mean':    global_mean,
    'global_std':     global_std,
    'energy':         energy,
    'quad_top_left':  top_left,
    'quad_top_right': top_right,
    'quad_bot_left':  bottom_left,
    'quad_bot_right': bottom_right,
})

# Add the histograms to the same DataFrame
df_custom_features = pd.concat([df_custom_features, df_histograms], axis=1)

print("Custom feature dimensions:", df_custom_features.shape)


In [ ]:
# -----------------------------------------------------------
# 4.6 Feature Unification + Dimensionality Reduction (PCA)
# -----------------------------------------------------------
# Combine all the feature sets created in the previous steps:
#   - Normalized pixels (784 cols)
#   - Horizontal and vertical asymmetry (2 cols)
#   - Sobel filter (784 cols)
#   - HOG (36 cols with pixels_per_cell=14x14)
#   - Custom features: global stats + quadrants + histograms (23 cols)
# Approximate total: ~1,630 columns -> reduce with PCA

# 1. Concatenate all features into a single DataFrame
X_final = pd.concat([
    X_normalized,                                          # 784 normalized pixel columns
    X_fe[['horizontal_asymmetry', 'vertical_asymmetry']],  # 2 symmetry columns
    df_sobel,                                              # 784 Sobel edge columns
    df_hog,                                                # 36 HOG columns
    df_custom_features                                     # 23 statistics columns
], axis=1)

print("1. Dimensions of all combined features:", X_final.shape)

# 2. Split train and test respecting the original dataset partition
#    (the first 60,000 rows are train, the remaining 10,000 are test)
#    NOTE: we don't use train_test_split here to keep the official dataset split.
# _ml suffix to distinguish these from the CNN variables
X_train_ml = X_final.iloc[:60000]
X_test_ml  = X_final.iloc[60000:]
y_train_ml = y.iloc[:60000]
y_test_ml  = y.iloc[60000:]

# 3. Scaling with StandardScaler
#    IMPORTANT: fit_transform only on TRAIN to avoid data leakage.
#    Transform on TEST using the parameters learned on TRAIN.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ml)
X_test_scaled  = scaler.transform(X_test_ml)

# 4. Reduction with PCA (keep 95% of the explained variance)
#    PCA automatically determines how many components are needed.
#    Reducing dimensions improves training speed and can reduce overfitting.
pca = PCA(n_components=0.95, random_state=42)

# Fit PCA only on TRAIN (same principle as the scaler)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print("2. TRAIN dimensions after PCA:", X_train_pca.shape)
print("3. TEST dimensions after PCA:", X_test_pca.shape)


## 5. Classical Machine Learning Models + Ensemble


In [ ]:
# ===========================================================
# 5 & 6. CLASSICAL MACHINE LEARNING MODELS + ENSEMBLE
# ===========================================================

# -----------------------------------------------------------
# 6.1 Random Forest
# -----------------------------------------------------------
# What is it? A collection of decision trees trained on random subsets
# of data and features. The final prediction is the most voted class.
# Why? Robust, handles high dimensionality and noise well.
# n_estimators=100: 100 trees in the forest
# n_jobs=-1: use all CPU cores for parallel training
print("--- Training Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_pca, y_train_ml)

rf_preds = rf_model.predict(X_test_pca)
print("Random Forest Accuracy:", accuracy_score(y_test_ml, rf_preds))
print(classification_report(y_test_ml, rf_preds))

# -----------------------------------------------------------
# 6.2 Logistic Regression
# -----------------------------------------------------------
# What is it? A linear model that computes the probability of each class via
# the softmax function (the multiclass extension of logistic regression).
# Why? Fast, interpretable, and a good baseline to compare against more complex models.
# max_iter=1000: increase the iteration limit to ensure convergence with PCA
print("\n--- Training Logistic Regression ---")
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train_pca, y_train_ml)

lr_preds = lr_model.predict(X_test_pca)
print("Logistic Regression Accuracy:", accuracy_score(y_test_ml, lr_preds))
print(classification_report(y_test_ml, lr_preds))

# -----------------------------------------------------------
# 7. Ensemble: Voting Classifier (Soft Voting)
# -----------------------------------------------------------
# What is it? Combines the predictions of Random Forest and Logistic Regression.
# voting='soft': averages the PROBABILITIES of each class (not just the majority vote).
# Soft voting usually outperforms hard voting when the models are well calibrated.
print("\n--- Training Ensemble (Voting Classifier) ---")
voting_model = VotingClassifier(
    estimators=[
        ('rf', rf_model),   # Model 1: Random Forest
        ('lr', lr_model)    # Model 2: Logistic Regression
    ],
    voting='soft',          # Average probabilities instead of binary votes
    n_jobs=-1
)

voting_model.fit(X_train_pca, y_train_ml)
voting_preds = voting_model.predict(X_test_pca)

print("Voting Classifier Accuracy:", accuracy_score(y_test_ml, voting_preds))
print(classification_report(y_test_ml, voting_preds))


## 6. Deep Learning — Convolutional Neural Network (CNN)


In [ ]:
# ===========================================================
# 8. DEEP LEARNING - CNN (Convolutional Neural Network)
# ===========================================================

# 1. Data preparation for the CNN
#    Classical ML models receive flat vectors (784 numbers),
#    but CNNs need 2D matrices (28x28) with a color channel.
#    Fashion-MNIST is grayscale -> 1 channel (RGB would have 3 channels).
#    We use the _cnn suffix to avoid overwriting the classical ML pipeline variables.

X_train_raw = df_train.drop('label', axis=1).values  # 60,000 x 784 pixels
y_train_cnn = df_train['label'].values                # Training labels

X_test_raw  = df_test.drop('label', axis=1).values   # 10,000 x 784 pixels
y_test_cnn  = df_test['label'].values                 # Test labels

# 2. Reshape to (num_samples, height, width, channels) format and normalize to [0,1]
X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1) / 255.0
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1) / 255.0

print(f"X_train dimensions ready for CNN: {X_train_cnn.shape}")
print(f"X_test  dimensions ready for CNN: {X_test_cnn.shape}")


In [ ]:
# -----------------------------------------------------------
# CNN Architecture
# -----------------------------------------------------------
# We use a simple CNN with two convolutional blocks followed by dense layers.
# Chosen architecture:
#
#   [Conv2D(32)] -> [MaxPool] -> [Dropout(0.25)]
#       Detects simple features: edges, lines
#
#   [Conv2D(64)] -> [MaxPool] -> [Dropout(0.25)]
#       Detects more complex features: shapes, textures
#
#   [Flatten] -> [Dense(128)] -> [Dropout(0.5)] -> [Dense(10, softmax)]
#       Final classification into 10 categories

model_cnn = Sequential([

    # --- Block 1: Extraction of simple local features ---
    # kernel_size=(3,3): 3x3 pixel window to detect patterns
    # activation='relu': activation function that removes negative values
    # input_shape=(28,28,1): shape of each input image
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),

    # Reduce dimensions by half (from 26x26 to 13x13) by taking the max of each 2x2 zone
    # Makes the model more efficient and gives it some translation invariance
    MaxPooling2D(pool_size=(2, 2)),

    # Randomly deactivates 25% of the neurons during training
    # Prevents the model from memorizing the dataset (overfitting)
    Dropout(0.25),

    # --- Block 2: Extraction of more abstract features ---
    # 64 filters (double the previous block): captures combinations of earlier patterns
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    # --- Transition to the dense classifier ---
    # Converts the 2D output of the feature maps into a 1D vector
    Flatten(),

    # Dense layer of 128 neurons: learns global combinations of all features
    Dense(128, activation='relu'),

    # Higher dropout (50%) before the output layer for stronger regularization
    Dropout(0.5),

    # Output layer: 10 neurons (one per class), softmax gives probabilities that sum to 1
    Dense(10, activation='softmax')
])

# Architecture summary: number of parameters per layer
model_cnn.summary()


In [ ]:
# -----------------------------------------------------------
# CNN Compilation and Training
# -----------------------------------------------------------

# - optimizer='adam': adaptive gradient descent (efficient and robust)
# - loss='sparse_categorical_crossentropy': standard loss for multiclass classification
#   (we use 'sparse' because the labels are integers, not one-hot encoded)
# - metrics=['accuracy']: monitor accuracy during training
model_cnn.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# - batch_size=128: process 128 images before updating the weights
# - epochs=10: 10 full passes over the training dataset
# - validation_split=0.2: reserve 20% of train (12,000 images) to validate each epoch
print("Starting CNN training...")
history = model_cnn.fit(X_train_cnn, y_train_cnn,
                        batch_size=128,
                        epochs=10,
                        validation_split=0.2,
                        verbose=1)

# -----------------------------------------------------------
# Training Curves: Accuracy and Loss per epoch
# -----------------------------------------------------------
# These curves let us detect overfitting:
#   - If val_accuracy keeps rising along with train_accuracy -> the model generalizes well
#   - If val_accuracy plateaus or drops while train keeps rising -> there is overfitting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'],     label='Train',      linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2, linestyle='--')
axes[0].set_title('Accuracy per epoch', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'],     label='Train',      linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2, linestyle='--')
axes[1].set_title('Loss per epoch', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('CNN Training Curves', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 7. Model Evaluation and Comparison


In [ ]:
# ===========================================================
# 9. MODEL EVALUATION AND COMPARISON
# ===========================================================

# NOTE on label variables:
#   y_test_ml  -> test set labels for the classical ML models
#   y_test_cnn -> test set labels for the CNN
#   Both contain the same 10,000 values, but we keep them separate
#   to avoid confusion between the two pipelines.

# 1. CNN predictions
#    model_cnn.predict() returns probabilities for each of the 10 classes.
#    np.argmax(..., axis=1) selects the index (class) with the highest probability.
y_pred_cnn_prob = model_cnn.predict(X_test_cnn)
y_pred_cnn = np.argmax(y_pred_cnn_prob, axis=1)

# 2. Parallel lists: models, predictions, and their corresponding true labels
models           = ['Random Forest', 'Logistic Regression', 'Ensemble (Voting)', 'Deep Learning (CNN)']
predictions      = [rf_preds,         lr_preds,              voting_preds,        y_pred_cnn]
true_labels      = [y_test_ml,        y_test_ml,             y_test_ml,            y_test_cnn]

# 3. Compute metrics for each model
#    F1-Score with average='macro': arithmetic mean of the F1 score for each class
#    (gives equal weight to all classes, appropriate for balanced datasets)
accuracies = [accuracy_score(y_t, y_p) for y_t, y_p in zip(true_labels, predictions)]
f1_scores  = [f1_score(y_t, y_p, average='macro') for y_t, y_p in zip(true_labels, predictions)]

# 4. Comparison table sorted from highest to lowest Accuracy
df_comparison = pd.DataFrame({
    'Model':             models,
    'Accuracy':          accuracies,
    'F1-Score (Macro)':  f1_scores
})

print("--- Performance Comparison Table ---")
display(df_comparison.sort_values(by='Accuracy', ascending=False))


In [ ]:
# Horizontal bar chart to visually compare the Accuracy of all models
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    x='Accuracy',
    y='Model',
    data=df_comparison.sort_values('Accuracy', ascending=False),
    palette='magma'
)

# Add the exact value at the end of each bar for easier numeric comparison
for p in ax.patches:
    ax.annotate(f"{p.get_width():.4f}",
                (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center',
                xytext=(5, 0),
                textcoords='offset points',
                fontsize=11, fontweight='bold')

plt.title('Model Comparison: Classical Machine Learning vs Deep Learning (CNN)', fontsize=14, pad=15)
plt.xlabel('Accuracy on the Test Set', fontsize=12)
plt.ylabel('Algorithm', fontsize=12)
# Adjust the X axis so the differences are visually clearer
plt.xlim(0.75, 1.0)

plt.show()


In [ ]:
# Confusion Matrices: Classical ML (Ensemble) vs Deep Learning (CNN)
# What are they for? They show which classes get confused with each other.
# Rows = true class | Columns = model prediction
# Main diagonal = correct predictions
# Off-diagonal errors = confusions between similar classes

# class_names is already defined in the Imports cell
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Ensemble matrix (Classical ML)
cm_voting = confusion_matrix(y_test_ml, voting_preds)
sns.heatmap(cm_voting, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix: Ensemble (Classical ML)', fontsize=13)
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Prediction')

# CNN matrix (Deep Learning)
cm_cnn = confusion_matrix(y_test_cnn, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Confusion Matrix: CNN (Deep Learning)', fontsize=13)
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Prediction')

plt.tight_layout()
plt.show()


## 8. Generative AI


In [ ]:
# ===========================================================
# GENERATIVE AI: Loading and Preparing Metadata
# ===========================================================

# 1. Load the JSON with synthetic metadata for the test set (10,000 products)
df_json = pd.read_json(FOLDER_PATH + "fashion_mnist_metadata.json")

# 2. Expand the 'metadata' column (contains a dictionary per row)
#    pd.json_normalize turns the dictionary fields into separate columns
df_metadata = pd.json_normalize(df_json['metadata'])

# Join the clean JSON with the expanded metadata
df_json_clean = pd.concat([df_json.drop(columns=['metadata']), df_metadata], axis=1)

# 3. Positional JOIN (by index) between the image test set and the metadata
df_test_joined = pd.concat([df_test.reset_index(drop=True),
                             df_json_clean.reset_index(drop=True)], axis=1)
df_test_joined = df_test_joined.loc[:, ~df_test_joined.columns.duplicated()]

# 4. Function that builds the full textual "context" for each product
#    We include every available field from the JSON so the LLM
#    has maximum information and can generate richer, more accurate descriptions.
def create_context(row):
    materials_str = ", ".join(row['materials']) if isinstance(row.get('materials'), list) else str(row.get('materials', ''))
    occasions_str  = ", ".join(row['occasions'])  if isinstance(row.get('occasions'),  list) else str(row.get('occasions',  ''))
    return (
        f"Category: {row['category']} ({row.get('gender', 'Unisex')}). "
        f"Brand: {row['brand']}. Model: {row['model']}. "
        f"Color: {row['color']}. Size: {row.get('size', 'N/A')}. "
        f"Price: ${row['price']}. "
        f"Rating: {row.get('rating', 'N/A')}/5 ({row.get('num_reviews', 0)} reviews). "
        f"Stock: {row.get('stock', 'N/A')} units. "
        f"Materials: {materials_str}. "
        f"Occasions: {occasions_str}."
    )

df_test_joined['context'] = df_test_joined.apply(create_context, axis=1)

print("JOIN completed. Dimensions:", df_test_joined.shape)
print("\nExample of generated context:")
print(df_test_joined['context'].iloc[0])


In [ ]:

# ===========================================================
# Automatic Description Generation with an Open Source LLM
# ===========================================================
# We use TinyLlama-1.1B-Chat: a lightweight open source language model,
# runnable on a regular laptop or in Colab without needing a high-end GPU.
# torch_dtype=torch.float16: uses 16-bit precision (less RAM usage)
# device_map="auto": places the model on GPU if available, otherwise on CPU

print("Loading the Open Source LLM (TinyLlama)...")
pipe = pipeline("text-generation",
                model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                torch_dtype=torch.float16,
                device_map="auto")

def generate_ai_description(context):
    """
    Generates a professional e-commerce description from the product metadata.
    Uses TinyLlama's chat format with a role system (system + user).
    """
    messages = [
        # 'system' role: defines the LLM's behavior (acts as a marketing expert)
        {"role": "system", "content": "You are a fashion marketing expert. Write a short, persuasive 2-sentence product description for an e-commerce store based on the product data. Only return the description."},
        # 'user' role: provides the specific product data
        {"role": "user", "content": f"Product data: {context}"}
    ]
    # Convert the messages into the prompt format expected by TinyLlama
    prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    try:
        # max_new_tokens=60: limit the response length to keep it concise
        # temperature=0.7: controls creativity (0=deterministic, 1=very creative)
        outputs = pipe(prompt, max_new_tokens=60, do_sample=True, temperature=0.7)
        # Extract only the assistant's response part (after the <|assistant|> token)
        return outputs[0]["generated_text"].split("<|assistant|>")[-1].strip()
    except Exception as e:
        return f"Error: {e}"

# Select exactly 1 sample from each of the 10 classes
df_10_samples = df_test_joined.groupby('category').sample(n=1, random_state=42).copy()

print("Generating the 10 descriptions...\n")
df_10_samples['ai_description'] = df_10_samples['context'].apply(generate_ai_description)

# Show the formatted results
for index, row in df_10_samples.iterrows():
    print(f" CLASS: {row['category'].upper()} |  {row['brand']} {row['model']}")
    print(f" AI DESCRIPTION: {row['ai_description']}\n" + "-"*80)

# IMPORTANT: free the model from RAM so the embedding model can be loaded next
# Language models take up several GB; without freeing memory, the next cell would fail.
print("\n CLEARING THE MODEL FROM RAM TO FREE UP SPACE...")
del pipe
gc.collect()  # Force the release of unreferenced Python objects
if torch.cuda.is_available():
    torch.cuda.empty_cache()  # Clear the GPU cache
print(" RAM freed.")


In [ ]:
# ===========================================================
# Semantic Search with Embeddings (SentenceTransformers)
# ===========================================================
# What are embeddings?
#   Vector (numeric) representations of the meaning of a text.
#   Texts with similar meanings have similar vectors (high cosine similarity).
#
# What is all-MiniLM-L6-v2?
#   A lightweight, fast SentenceTransformers model, specifically trained
#   to compute semantic similarity between texts.

# !pip install sentence-transformers  # Uncomment if the library is not installed

print("Loading the Embeddings model (all-MiniLM-L6-v2)...")
encoder = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Vectorize the contexts of the 10,000 test set products
#    convert_to_tensor=True: returns PyTorch tensors (faster for search)
#    show_progress_bar=True: shows progress while processing the 10,000 rows
print("Encoding the catalog, this will take a few seconds...")
catalog_embeddings = encoder.encode(df_test_joined['context'].tolist(),
                                     convert_to_tensor=True, show_progress_bar=True)

def search_similar(query, top_k=5):
    """
    Finds the top_k products that are semantically most similar to the user's query.
    Parameters:
        query  : natural language search text
        top_k  : number of results to return (default 5)
    """
    # Vectorize the user's query with the same model
    query_emb = encoder.encode(query, convert_to_tensor=True)

    # Compute cosine similarity between the query and all products in the catalog
    # util.semantic_search returns the top_k closest matches, sorted by score
    hits = util.semantic_search(query_emb, catalog_embeddings, top_k=top_k)[0]

    print(f"\n SEARCH: '{query}'\n" + "="*70)
    for i, hit in enumerate(hits):
        idx = hit['corpus_id']   # Product index in the catalog
        item = df_test_joined.iloc[idx]
        print(f"TOP {i+1} (Score: {hit['score']:.4f}) | {item['category'].upper()}")
        print(f"Product: {item['brand']} {item['model']} | ${item['price']}")
        print(f"Context: {item['context']}\n" + "-"*70)

# 3 different searches to demonstrate the system's versatility
queries = [
    "Comfortable cotton t-shirt for daily use",
    "Elegant leather bag for office and meetings",
    "Warm winter coat for snowy days"
]

for q in queries:
    search_similar(q, top_k=5)
